In [1]:
import os
import sys
import importlib
import math
import time

import itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm as tqdm_auto
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import pytorch_lightning as pl

from IPython.display import clear_output

In [2]:
sys.path.append("../model/")
import utrdata_cl as utrdata
from legnet import LegNetClassifier
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr5"

In [4]:
if utr_type == "utr5":
    seqsize = 50
    MODEL_PATH = "saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    seqsize = 240
    MODEL_PATH = "saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
PATH_FROM = f"./{utr_type.upper()}_zscores_replicateagg.csv"
df = pd.read_csv(PATH_FROM)

In [6]:
num_classes = df["cell_type"].unique().shape[0]
num_classes

5

In [7]:
batch_size = 1024

In [8]:
num_workers = 32

In [9]:
test_set = utrdata.UTRData(
    df=df,
    construct_type=utr_type,
    features=("sequence", "positional", "conditions"),  # ("sequence", "conditions", "positional", "revcomp")
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False,
)

model = RNARegressor.load_from_checkpoint(MODEL_PATH)
trainer = pl.Trainer(
    callbacks=[pl.callbacks.TQDMProgressBar(refresh_rate=0.5)],
    logger=False,
    accelerator="gpu",
    devices=[0],
    deterministic=True,
    num_sanity_val_steps=0,
)
prediction = trainer.predict(model=model, dataloaders=dl_test)
test_pred, test_real = zip(*prediction)
test_pred = torch.concat(test_pred).numpy()
test_real = torch.concat(test_real).numpy()
test_df = df.copy()
test_df["pred_delta"] = test_pred[:, 0]
test_df["pred_mass_center"] = test_pred[:, 1]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [10]:
test_df.to_csv(f"{utr_type.upper()}_preds.csv", index=False)

---